In [ ]:
# #Test search serper API
# import http.client
# import json

# # Khởi tạo kết nối
# conn = http.client.HTTPSConnection("google.serper.dev")

# # Payload tìm kiếm - bạn có thể thay đổi "q" tùy ý
# payload = json.dumps({
#   "q": "apple inc",
#   "gl": "us", # Quốc gia (tùy chọn)
#   "hl": "en"  # Ngôn ngữ (tùy chọn)
# })

# # Header chứa API Key của bạn
# # Lưu ý: Mình sử dụng key thứ 2 bạn vừa cung cấp
# headers = {
#   'X-API-KEY': '8de3e16fd703d143c9e5d261c358a171d353e641',
#   'Content-Type': 'application/json'
# }

# # Gửi request
# conn.request("POST", "/search", payload, headers)

# # Nhận phản hồi
# res = conn.getresponse()
# data = res.read()

# # Giải mã và hiển thị dưới dạng JSON đẹp
# result = json.loads(data.decode("utf-8"))
# print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "searchParameters": {
    "q": "apple inc",
    "gl": "us",
    "hl": "en",
    "type": "search",
    "engine": "google"
  },
  "knowledgeGraph": {
    "title": "Apple",
    "imageUrl": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcT5ITHsQzdzkkFWKinRe1Y4FUbC_Vy3R_M&s=0",
    "description": "Apple Inc. is an American multinational technology company headquartered in Cupertino, California, in Silicon Valley, and known for consumer electronics, software and online services.",
    "descriptionSource": "Wikipedia",
    "descriptionLink": "https://en.wikipedia.org/wiki/Apple_Inc.",
    "attributes": {
      "Designated CEO": "John Ternus (Sep 1, 2026–)",
      "Customer service": "1 (800) 275-2273",
      "Founders": "Steve Jobs, Steve Wozniak, and Ronald Wayne",
      "Founded": "April 1, 1976, Los Altos, CA",
      "Headquarters": "Cupertino, CA",
      "COO": "Sabih Khan",
      "CEO": "Tim Cook (Aug 24, 2011–Sep 1, 2026)"
    }
  },
  "organic": [
    {
      "title": "Apple

# claim to entity using ReFined

In [ ]:
# !pip install -q https://github.com/amazon-science/ReFinED/archive/refs/tags/V1.zip

In [ ]:
# import transformers

# # --- VÁ LỖI 1: KHỞI TẠO TOKENIZER ---
# original_init = transformers.tokenization_utils_base.PreTrainedTokenizerBase.__init__

# def patched_init(self, *args, **kwargs):
#     kwargs.pop('add_special_tokens', None)
#     kwargs.pop('add_prefix_space', None)
#     original_init(self, *args, **kwargs)

# transformers.tokenization_utils_base.PreTrainedTokenizerBase.__init__ = patched_init

# # --- VÁ LỖI 2: HÀM ENCODE_PLUS BỊ XÓA ---
# if not hasattr(transformers.tokenization_utils_base.PreTrainedTokenizerBase, 'encode_plus'):
#     def patched_encode_plus(self, text, text_pair=None, **kwargs):
#         # Chuyển hướng lệnh gọi cũ sang cách gọi mới của transformers
#         if text_pair is not None:
#             return self.__call__(text, text_pair, **kwargs)
#         return self.__call__(text, **kwargs)

#     transformers.tokenization_utils_base.PreTrainedTokenizerBase.encode_plus = patched_encode_plus

# # --- KHỞI TẠO VÀ TEST REFINED ---
# from refined.inference.processor import Refined

# # Khởi tạo mô hình
# refined = Refined.from_pretrained(
#     model_name='wikipedia_model_with_numbers',
#     entity_set="wikipedia"
# )



In [ ]:
# # Chạy test thử luôn đoạn văn bản
# spans = refined.process_text("England won the FIFA World Cup in 1966.")
# print(spans)

# Truy vấn dbPedia

In [ ]:
# import requests
# import urllib.parse

# def query_dbpedia_one_hop(entity_title):
#     """
#     Truy vấn DBpedia để lấy các bộ ba 1-hop (Chỉ lấy tiếng Anh)
#     """
#     entity_formatted = urllib.parse.quote(entity_title.replace(" ", "_"))
#     resource_uri = f"http://dbpedia.org/resource/{entity_formatted}"

#     # Đã cập nhật dòng FILTER cuối cùng để chỉ lấy tiếng Anh (en)
#     query = f"""
#     SELECT ?s ?p ?o WHERE {{
#         {{
#             <{resource_uri}> ?p ?o .
#             BIND(<{resource_uri}> AS ?s)
#         }}
#         UNION
#         {{
#             ?s ?p <{resource_uri}> .
#             BIND(<{resource_uri}> AS ?o)
#         }}
#         FILTER (!regex(str(?p), "wikiPage|subject|type|sameAs|Abstract|thumbnail|depiction", "i"))
#         FILTER (isIRI(?o) || (isLiteral(?o) && (lang(?o) = "" || langMatches(lang(?o), "en"))))
#     }}
#     LIMIT 200
#     """

#     url = "http://dbpedia.org/sparql"
#     params = {
#         "query": query,
#         "format": "json"
#     }

#     headers = {
#         "User-Agent": "FactCheckingBot/1.0"
#     }

#     response = requests.get(url, params=params, headers=headers)

#     triples = []
#     if response.status_code == 200:
#         results = response.json()
#         for result in results["results"]["bindings"]:
#             s = result["s"]["value"].split('/')[-1]
#             p = result["p"]["value"].split('/')[-1]
#             o = result["o"]["value"].split('/')[-1]

#             # Thay thế dấu gạch dưới bằng khoảng trắng cho dễ đọc nếu cần
#             triple_str = f"{s} -> {p} -> {urllib.parse.unquote(o)}"
#             triples.append(triple_str)
#         return triples
#     else:
#         print(f"Lỗi truy vấn DBpedia: Code {response.status_code}")
#         return []

# # Test lại
# entity_title = "England national football team"
# extracted_triples = query_dbpedia_one_hop(entity_title)

# for i, triple in enumerate(extracted_triples[:10]):
#     print(f"{i+1}. {triple}")

# Hàm reranking truy vấn để tìm ra top k các truy vấn/snippet liên quan nhất

In [ ]:
# from sentence_transformers.cross_encoder import CrossEncoder
# import numpy as np

# # 1. Khởi tạo mô hình Cross-Encoder đúng như paper sử dụng
# print("Đang tải mô hình ms-marco-MiniLM-L-6-v2...")
# cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# def rerank_evidence(claim, evidence_list, top_k=5):
#     """
#     Hàm ghép cặp Claim và Evidence, sau đó chấm điểm và trả về Top K
#     """
#     if not evidence_list:
#         return []

#     # 2. Tạo danh sách các cặp (Claim, Evidence)
#     sentence_pairs = [[claim, ev] for ev in evidence_list]

#     # 3. Cho mô hình chấm điểm mức độ liên quan ngữ nghĩa
#     scores = cross_encoder.predict(sentence_pairs)

#     # 4. Ghép điểm số với bằng chứng tương ứng
#     scored_evidence = list(zip(scores, evidence_list))

#     # 5. Sắp xếp giảm dần theo điểm số (x[0] là điểm score)
#     scored_evidence.sort(key=lambda x: x[0], reverse=True)

#     # 6. Trả về đúng Top K (K=5 như paper)
#     top_evidence = [ev for score, ev in scored_evidence[:top_k]]

#     return top_evidence

# # ==========================================
# # TEST GHÉP NỐI VỚI DỮ LIỆU BƯỚC TRƯỚC
# # ==========================================

# claim = "England won the FIFA World Cup in 1966."

# # Giả sử đây là danh sách extracted_triples bạn lấy được từ DBpedia ở bước trước
# extracted_triples = [
#     "England_national_football_team -> rdf-schema#label -> England national football team",
#     "England_national_football_team -> type -> NationalFootballTeam",
#     "1966_FIFA_World_Cup -> champion -> England_national_football_team", # triple này rất khớp ngữ nghĩa
#     "England_national_football_team -> manager -> Gareth_Southgate",
#     "England_national_football_team -> captain -> Harry_Kane",
#     "England_national_football_team -> ground -> Wembley_Stadium",
#     "FIFA_World_Cup -> winner -> England_national_football_team" # triple này cũng khá khớp
# ]

# print(f"\nĐang Re-rank {len(extracted_triples)} triples...")
# top_5_triples = rerank_evidence(claim, extracted_triples, top_k=5)

# print("\n--- TOP 5 BẰNG CHỨNG TỐT NHẤT ---")
# for i, triple in enumerate(top_5_triples):
#     print(f"{i+1}. {triple}")

In [ ]:
# import pickle
# import pprint
# import pandas as pd

# file_path = '/content/factkg_test.pickle'

# try:
#     with open(file_path, 'rb') as f:
#         factkg_data = pickle.load(f)

#     print(f"✅ Đã tải thành công! Tổng số lượng mẫu: {len(factkg_data)}")
#     print("\n" + "="*40)
#     print("MẪU DỮ LIỆU ĐẦU TIÊN (SAMPLE 0)")
#     print("="*40)

#     # Kiểm tra xem dữ liệu được lưu dưới dạng List hay Pandas DataFrame
#     if isinstance(factkg_data, list):
#         sample = factkg_data[0]
#         pprint.pprint(sample, sort_dicts=False)
#     elif isinstance(factkg_data, pd.DataFrame):
#         sample = factkg_data.iloc[0].to_dict()
#         pprint.pprint(sample, sort_dicts=False)
#     else:
#         print(f"Kiểu dữ liệu lạ: {type(factkg_data)}")
#         print(factkg_data)

# except FileNotFoundError:
#     print(f" Lỗi: Không tìm thấy file tại '{file_path}'.")
#     print("Bạn kiểm tra lại xem đã upload file factkg_test.pickle lên thư mục /content/ của Colab chưa nhé!")
# except Exception as e:
#     print(f" Có lỗi xảy ra khi đọc file: {e}")

## Full pipe line hybrid fact-checking

In [ ]:
!pip install -q https://github.com/amazon-science/ReFinED/archive/refs/tags/V1.zip

     - 202.4 kB 6.7 MB/s 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.5 MB/s eta 0:00:00


In [ ]:
# ==========================================
# 0. RÚT PHÍCH CẮM HỆ THỐNG CẢNH BÁO
# ==========================================
import warnings
import logging
import os
import sys

def block_all_warnings(*args, **kwargs):
    pass
warnings.showwarning = block_all_warnings
warnings.filterwarnings("ignore")

logging.getLogger().setLevel(logging.CRITICAL)
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ==========================================
# 1. IMPORT THƯ VIỆN & CẤU HÌNH CƠ BẢN
# ==========================================
import re
import json
import time
import pickle
import urllib.parse
import requests
import csv
import concurrent.futures
from tqdm import tqdm
from collections import defaultdict
import transformers

# SAFE PATCH TRANSFORMERS & REFINED
if not getattr(transformers, '_is_patched', False):
    original_init = transformers.tokenization_utils_base.PreTrainedTokenizerBase.__init__
    def patched_init(self, *args, **kwargs):
        kwargs.pop('add_special_tokens', None)
        kwargs.pop('add_prefix_space', None)
        original_init(self, *args, **kwargs)
    transformers.tokenization_utils_base.PreTrainedTokenizerBase.__init__ = patched_init

    if not hasattr(transformers.tokenization_utils_base.PreTrainedTokenizerBase, 'encode_plus'):
        def patched_encode_plus(self, text, text_pair=None, **kwargs):
            if text_pair is not None:
                return self.__call__(text, text_pair, **kwargs)
            return self.__call__(text, **kwargs)
        transformers.tokenization_utils_base.PreTrainedTokenizerBase.encode_plus = patched_encode_plus
    transformers._is_patched = True

# ==========================================
# 2. KHỞI TẠO MODEL & QUẢN LÝ API KEY
# ==========================================
print("▶ Đang khởi tạo các mô hình và API... (Vui lòng đợi)")

from refined.inference.processor import Refined
refined = Refined.from_pretrained(model_name='wikipedia_model_with_numbers', entity_set="wikipedia")

from sentence_transformers.cross_encoder import CrossEncoder
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# >>> SỬ DỤNG DEEPSEEK API <<<
from openai import OpenAI
client = OpenAI(
  base_url="https://api.deepseek.com",
  api_key="sk-5e296cec3b1147039d3eb0465b06829e"
)

# Quản lý 38 API Keys của Serper (Xoay vòng mỗi 2400 requests)
class SerperKeyManager:
    def __init__(self):
        self.keys = [
            "8de3e16fd703d143c9e5d261c358a171d353e641", "838cfa7e6b0c2e726d16bc78d0c2da2043b2726d",
            "87d1bf1e69243c6ea4c8a41c99f80c5f8eed44a0", "9b0e3bd6c2d3eaeafda2b8c66487dfa53193418b",
            "42cca14a5edddc1f8a59e5ba800117dc8ffbd72a", "521e58ee17222263aed162ef04a853f2cd85392d",
            "cd833a90e33314ce3c95e9fef0285f9026efc44b", "0ce3ed49364f1f5d11c9b1e6cb31442b21f45a7a",
            "722e94473682d9439a231413d34ddb63fa669cbd", "c13ce735895c4fa6964b3783cde76a134a92c327",
            "e48a7891538fd3b89e1bcb0a023e11c88b0b5752", "1e9feca2656e6d665e69a1751f6536da97a224b4",
            "90d780c7d067ee6af8eb3139b753eff5a77f46ae", "f7e3e1bc884d318014759e47a04e2fb52f94150c",
            "8fb253c7a06b07c06b9132eed7db7b177b52454f", "436811c27b328cad42066f7a748824a7b00e3e16",
            "c6192d4863aaed6bcf69aa0339bc833f15afa03f", "af9b0d0eb75ee006f47e5f4a4a5250e7e16d783f",
            "86307e2b456d10afd65ffd6666308414f3c05fba", "1bff686f282ae6e5a54c6a1c0e21ed3c0d1af081",
            "14f694f87b34575304760ab88601a9736931424e", "c1b3b23ced9bce9529d42401250c0daeb3a08ae0",
            "1a3546f5fcd9d9eac2e9ebb324310347e9c8de60", "d55f0758bce79e5dba8519bec974ee537061bbe9",
            "dc076a37bf027a00286d21443685a49d627c09af", "8331b883f0cf3e76cb2aa5e714011319e31abe37",
            "844b49547abbcb55ada16c9875279d7fe74187ef", "0ee15a039305266aa50e50d9b5e701262c45c0cf",
            "45c97517036473e1f38fdf95b735763cbf2e46e1", "dd9c7e12a27d670fd24036f00243062c9cc93f94",
            "be213b0aba9d8efd9b112dd176cff4e715845ecf", "5272634854374657872b32e2fa479d6a5f457cf6",
            "1129688ece00ce076ea94643e045ea95c336c0a1", "f41589ef6fb2b867668d4424b933ead370edd555",
            "7c8d4069fe8b2d52d256da20ffe23add8ce22007", "13484a33539b951c6dad54e14f546bc2dbb5db11",
            "5de176a5ca6def28b2ebce62b1dbf3691a94f993"
        ]
        self.current_idx = 0
        self.uses = 0
        self.max_uses = 2400

    def get_key(self):
        if self.uses >= self.max_uses:
            self.current_idx = (self.current_idx + 1) % len(self.keys)
            self.uses = 0
            print(f"\n[Serper API] Đã chuyển sang API Key số {self.current_idx + 1}")
        self.uses += 1
        return self.keys[self.current_idx]

key_manager = SerperKeyManager()

# ==========================================
# 3. HÀM CỐT LÕI TÌM KIẾM VÀ LLM
# ==========================================
dbpedia_cache = {}

def query_dbpedia_one_hop(entity_title, max_retries=3):
    entity_formatted = urllib.parse.quote(entity_title.replace(" ", "_"))
    resource_uri = f"http://dbpedia.org/resource/{entity_formatted}"
    query = f"""
    SELECT ?s ?p ?o WHERE {{
        {{ <{resource_uri}> ?p ?o . BIND(<{resource_uri}> AS ?s) }}
        UNION
        {{ ?s ?p <{resource_uri}> . BIND(<{resource_uri}> AS ?o) }}
        FILTER (!regex(str(?p), "wikiPage|subject|type|sameAs|Abstract|thumbnail|depiction", "i"))
        FILTER (isIRI(?o) || (isLiteral(?o) && (lang(?o) = "" || langMatches(lang(?o), "en"))))
    }} LIMIT 100
    """
    for _ in range(max_retries):
        try:
            response = requests.get("http://dbpedia.org/sparql", params={"query": query, "format": "json"}, headers={"User-Agent": "Bot/1.0"}, timeout=10)
            if response.status_code == 200:
                return [f"{res['s']['value'].split('/')[-1]} -> {res['p']['value'].split('/')[-1]} -> {urllib.parse.unquote(res['o']['value'].split('/')[-1])}" for res in response.json()["results"]["bindings"]]
            time.sleep(1)
        except:
            time.sleep(2)
    return []

def get_triples_for_entity(ent):
    if ent in dbpedia_cache: return ent, dbpedia_cache[ent]
    return ent, query_dbpedia_one_hop(ent)

def rerank_evidence(claim, evidence_list, top_k=5):
    if not evidence_list: return []
    scores = cross_encoder.predict([[claim, ev] for ev in evidence_list])
    scored = sorted(zip(scores, evidence_list), key=lambda x: x[0], reverse=True)
    return [ev for _, ev in scored[:top_k]]

def search_web_snippets_serper(query, max_res=10, max_retries=3):
    snippets = []
    url = "https://google.serper.dev/search"
    payload = json.dumps({"q": query, "num": max_res})

    for attempt in range(max_retries):
        headers = {'X-API-KEY': key_manager.get_key(), 'Content-Type': 'application/json'}
        try:
            response = requests.post(url, headers=headers, data=payload, timeout=15)
            if response.status_code == 200:
                data = response.json()
                if "organic" in data:
                    snippets = [item["snippet"] for item in data["organic"] if "snippet" in item]
                break
            time.sleep(2)
        except:
            time.sleep(2 ** attempt)
    return snippets

# >>> CƠ CHẾ ADAPTIVE BATCHING + REGEX BÀN TAY SẮT <<<
def llm_batch_classify(batch_data, stage="KG", max_retries=3):
    if not batch_data: return {}

    final_result = {}
    queue = [batch_data.copy()] # Hàng đợi chứa các batch cần xử lý

    system_prompt = """You are an expert fact-checker evaluating claims.
You must return a mapping of claim_id to EXACTLY ONE of these labels: "Supported", "Refuted", or "Not Enough Info".
CRITICAL RULES:
1. Output ONLY format "claim_id": "Label"
2. DO NOT output explanations."""

    while queue:
        current_batch = queue.pop(0)
        if not current_batch: continue

        user_prompt = f"""Stage: {stage}
INPUT DATA:
{json.dumps(current_batch, indent=2)}

Format MUST strictly match this example exactly:
{{
  "claim_id_X": "Supported",
  "claim_id_Y": "Not Enough Info"
}}"""

        success = False
        for attempt in range(max_retries):
            try:
                completion = client.chat.completions.create(
                    model="deepseek-v4-flash",
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    temperature=0.0,
                    max_tokens=4000,
                )

                text = completion.choices[0].message.content.strip()
                if not text: raise ValueError("DeepSeek trả về chuỗi rỗng!")

                parsed = {}
                # BƯỚC 1: Thử parse JSON chuẩn mực trước
                try:
                    parsed = json.loads(text)
                except:
                    # BƯỚC 2: JSON lỗi do đứt gãy? Kích hoạt "Bàn Tay Sắt" Regex
                    pattern = r'"(c_\d+)"\s*:\s*"(Supported|Refuted|Not Enough Info)"'
                    matches = re.findall(pattern, text, re.IGNORECASE)

                    if matches:
                        # Chuẩn hóa lại format chữ hoa/thường cho chắc cốp
                        for cid, label in matches:
                            lbl_lower = label.lower()
                            if "supported" in lbl_lower: parsed[cid] = "Supported"
                            elif "refuted" in lbl_lower: parsed[cid] = "Refuted"
                            else: parsed[cid] = "Not Enough Info"
                    else:
                        raise ValueError(f"Regex cũng bó tay với text: {text[:80]}...")

                # Quét xem AI có bỏ sót (hoặc bị cụt) câu nào không
                missing = {}
                for cid in current_batch.keys():
                    if cid in parsed and parsed[cid] in ["Supported", "Refuted", "Not Enough Info"]:
                        final_result[cid] = parsed[cid]
                    else:
                        missing[cid] = current_batch[cid]

                # Nếu AI lười bỏ sót hoặc bị cắt cụt, gom các câu sót nhét lại vào hàng đợi
                if missing:
                    queue.insert(0, missing) # Ưu tiên xử lý luôn phần bị thiếu

                success = True
                break # Thoát vòng lặp retry vì đã thu hoạch được data

            except Exception as e:
                # Chỉ in lỗi để theo dõi, hệ thống vẫn tự sửa sai
                # print(f"\n[Lỗi DeepSeek {stage}] Thử lại {attempt+1}/{max_retries}: {str(e)[:120]}")
                time.sleep(2)

        # NẾU TẠCH 3 LẦN LIÊN TIẾP -> CHIA ĐÔI BATCH ĐỂ GIẢM TẢI
        if not success:
            keys = list(current_batch.keys())
            if len(keys) > 1:
                mid = len(keys) // 2
                part1 = {k: current_batch[k] for k in keys[:mid]}
                part2 = {k: current_batch[k] for k in keys[mid:]}
                queue.insert(0, part2)
                queue.insert(0, part1)
                # print(f"\n[Cứu hộ] Batch bị cụt/lỗi liên tục. Tự động chia đôi ({len(keys)} -> {len(part1)} và {len(part2)}) để thử lại.")
            else:
                # Nếu đã chia đến 1 câu mà vẫn tạch thì đành gán mặc định
                for cid in keys:
                    final_result[cid] = "Not Enough Info"

    return final_result


# ==========================================
# 4. KHỞI TẠO ĐƯỜNG DẪN VÀ ĐỌC DATA
# ==========================================
save_dir = '/content/drive/MyDrive/Data FactKG'
os.makedirs(save_dir, exist_ok=True)

f_kg_all = os.path.join(save_dir, '1_kg_evidence_all.json')
f_kg_empty = os.path.join(save_dir, '2_kg_empty.json')
f_kg_preds = os.path.join(save_dir, '3_kg_predictions.json')
f_web_needs = os.path.join(save_dir, '4_needs_web_search.json')
f_web_ev = os.path.join(save_dir, '5_web_evidence.json')
f_web_preds = os.path.join(save_dir, '6_web_predictions.json')
f_dbpedia_cache = os.path.join(save_dir, 'dbpedia_cache_backup.json')

file_path = os.path.join(save_dir, 'factkg_test.pickle')
with open(file_path, 'rb') as f:
    factkg_data = pickle.load(f)

# >>> THÔNG SỐ ĐA LUỒNG TỐI ƯU <<<
BATCH_SIZE_KG = 50
MAX_WORKERS_DBPEDIA = 10

BATCH_SIZE_LLM = 50
MAX_WORKERS_LLM = 5
MAX_WORKERS_SERPER = 10

ground_truths = {}
claim_types = {}
claims_dict = {}

for idx, (claim, metadata) in enumerate(factkg_data.items()):
    cid = f"c_{idx}"
    claims_dict[cid] = claim
    ground_truths[cid] = "Supported" if metadata['Label'][0] is True else "Refuted"
    claim_types[cid] = metadata.get('types', ['unknown'])
    claims_dict[f"{cid}_meta"] = metadata

def save_json(data, path):
    with open(path, 'w', encoding='utf-8') as f: json.dump(data, f, ensure_ascii=False, indent=2)

def load_json(path):
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f: return json.load(f)
    return {}

dbpedia_cache = load_json(f_dbpedia_cache)

# ==========================================
# PHASE 1: DUYỆT DATASET TRA DBPEDIA
# ==========================================
kg_evidence = load_json(f_kg_all)
kg_empty = load_json(f_kg_empty)

missing_kg_cids = [cid for cid in claims_dict.keys() if not cid.endswith('_meta') and cid not in kg_evidence and cid not in kg_empty]

if missing_kg_cids:
    print(f"\n▶ PHASE 1: Rút trích Knowledge Graph (DBpedia) cho {len(missing_kg_cids)} câu")
    for i in tqdm(range(0, len(missing_kg_cids), BATCH_SIZE_KG), desc="Tra DBpedia"):
        batch_cids = missing_kg_cids[i:i+BATCH_SIZE_KG]
        batch_entities_map = {}
        unique_entities_in_batch = set()

        for cid in batch_cids:
            claim = claims_dict[cid]
            metadata = claims_dict[f"{cid}_meta"]

            spans = refined.process_text(claim)
            entities = [getattr(s.entity, 'wikipedia_entity_title', None) for s in spans if hasattr(s, 'entity')]
            entities = [e for e in entities if e]

            if not entities and 'Entity_set' in metadata:
                entities = metadata['Entity_set']

            batch_entities_map[cid] = entities
            unique_entities_in_batch.update(entities)

        entities_to_fetch = [ent for ent in unique_entities_in_batch if ent not in dbpedia_cache]

        if entities_to_fetch:
            with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS_DBPEDIA) as executor:
                results = executor.map(get_triples_for_entity, entities_to_fetch)
                for ent, triples in results:
                    dbpedia_cache[ent] = triples
            save_json(dbpedia_cache, f_dbpedia_cache)

        for cid in batch_cids:
            claim = claims_dict[cid]
            triples = []
            for ent in batch_entities_map[cid]:
                triples.extend(dbpedia_cache.get(ent, []))

            top_triples = rerank_evidence(claim, triples, top_k=5)

            if top_triples:
                kg_evidence[cid] = {"claim": claim, "evidence": top_triples}
            else:
                kg_empty[cid] = {"claim": claim, "evidence": []}

        save_json(kg_evidence, f_kg_all)
        save_json(kg_empty, f_kg_empty)
else:
    print("\n▶ PHASE 1: Đã hoàn thành, bỏ qua...")


# ==========================================
# PHASE 2: LLM XỬ LÝ NHỮNG CÂU CÓ KG
# ==========================================
kg_preds = load_json(f_kg_preds)
cids_to_predict_kg = [cid for cid in kg_evidence.keys() if cid not in kg_preds]

if cids_to_predict_kg:
    print(f"\n▶ PHASE 2: LLM dự đoán nhãn KG (Batch={BATCH_SIZE_LLM}, Threads={MAX_WORKERS_LLM})")
    llm_batches = []
    for i in range(0, len(cids_to_predict_kg), BATCH_SIZE_LLM):
        batch_cids = cids_to_predict_kg[i:i+BATCH_SIZE_LLM]
        llm_batches.append({cid: kg_evidence[cid] for cid in batch_cids})

    def process_llm_batch_kg(batch):
        return llm_batch_classify(batch, stage="KG")

    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS_LLM) as executor:
        futures = [executor.submit(process_llm_batch_kg, batch) for batch in llm_batches]
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="LLM (KG)"):
            preds = future.result()
            kg_preds.update(preds)
            save_json(kg_preds, f_kg_preds)
else:
    print("\n▶ PHASE 2: Đã hoàn thành dự đoán KG, bỏ qua...")


# ==========================================
# GOM CÁC CÂU CẦN TRA WEB
# ==========================================
needs_web = load_json(f_web_needs)
for cid in list(kg_empty.keys()):
    if cid not in needs_web: needs_web[cid] = {"claim": claims_dict[cid]}

for cid, pred in kg_preds.items():
    if pred == "Not Enough Info" and cid not in needs_web:
        needs_web[cid] = {"claim": claims_dict[cid]}
save_json(needs_web, f_web_needs)


# ==========================================
# PHASE 3: TRA CỨU WEB QUA SERPER API
# ==========================================
web_evidence = load_json(f_web_ev)
cids_to_web = [cid for cid in needs_web.keys() if cid not in web_evidence]

if cids_to_web:
    print(f"\n▶ PHASE 3: Tra cứu Serper API cho {len(cids_to_web)} câu (Threads={MAX_WORKERS_SERPER})")

    def process_serper_search(cid):
        claim = needs_web[cid]["claim"]
        snippets = search_web_snippets_serper(claim, max_res=10)
        top_snippets = rerank_evidence(claim, snippets, top_k=5)
        return cid, claim, top_snippets

    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS_SERPER) as executor:
        futures = [executor.submit(process_serper_search, cid) for cid in cids_to_web]
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Tra Serper Web"):
            cid, claim, top_snippets = future.result()
            web_evidence[cid] = {"claim": claim, "evidence": top_snippets}
            save_json(web_evidence, f_web_ev)
else:
    print("\n▶ PHASE 3: Đã tra cứu đủ Serper API, bỏ qua...")


# ==========================================
# PHASE 4: LLM XỬ LÝ KẾT QUẢ TỪ WEB
# ==========================================
web_preds = load_json(f_web_preds)
cids_to_predict_web = [cid for cid in web_evidence.keys() if cid not in web_preds]

if cids_to_predict_web:
    print(f"\n▶ PHASE 4: LLM dự đoán nhãn Web (Batch={BATCH_SIZE_LLM}, Threads={MAX_WORKERS_LLM})")
    web_llm_batches = []
    for i in range(0, len(cids_to_predict_web), BATCH_SIZE_LLM):
        batch_cids = cids_to_predict_web[i:i+BATCH_SIZE_LLM]
        web_llm_batches.append({cid: web_evidence[cid] for cid in batch_cids})

    def process_llm_batch_web(batch):
        return llm_batch_classify(batch, stage="Web Fallback")

    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS_LLM) as executor:
        futures = [executor.submit(process_llm_batch_web, batch) for batch in web_llm_batches]
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="LLM (Web)"):
            preds = future.result()
            web_preds.update(preds)
            save_json(web_preds, f_web_preds)
else:
    print("\n▶ PHASE 4: Đã hoàn thành dự đoán Web, bỏ qua...")


# ==========================================
# PHASE 5: TỔNG HỢP VÀ TÍNH TOÁN ĐỘ CHÍNH XÁC (ACCURACY)
# ==========================================
print("\n▶ PHASE 5: Đánh giá độ chính xác (Accuracy)")
final_preds = {}

for cid in ground_truths.keys():
    kg_lbl = kg_preds.get(cid, "Not Enough Info")
    if kg_lbl in ["Supported", "Refuted"]:
        final_preds[cid] = kg_lbl
    else:
        wb_lbl = web_preds.get(cid, "Refuted")
        final_preds[cid] = wb_lbl if wb_lbl in ["Supported", "Refuted"] else "Refuted"

total = len(ground_truths)
correct = 0
type_stats = defaultdict(lambda: {"total": 0, "correct": 0})

csv_out = os.path.join(save_dir, 'final_evaluation.csv')
with open(csv_out, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['claim_id', 'claim', 'true_label', 'predicted_label', 'is_correct', 'types'])

    for cid, true_label in ground_truths.items():
        pred_label = final_preds[cid]
        is_correct = (pred_label == true_label)

        if is_correct: correct += 1

        c_types = claim_types[cid]
        for t in c_types:
            type_stats[t]["total"] += 1
            if is_correct: type_stats[t]["correct"] += 1

        writer.writerow([cid, claims_dict[cid], true_label, pred_label, is_correct, ",".join(c_types)])

print("\n" + "="*40)
print(f"🏆 TỔNG KẾT ĐÁNH GIÁ TRÊN FACTKG")
print("="*40)
print(f"Tổng số mẫu đã test: {total}")
print(f"Tổng độ chính xác (Overall Accuracy): {correct/total:.2%}\n")

print("Độ chính xác theo từng loại lập luận (Reasoning Types):")
for t, st in type_stats.items():
    if st["total"] > 0:
        acc = st["correct"] / st["total"]
        print(f" - {t.capitalize()}: {acc:.2%} ({st['correct']}/{st['total']})")

print(f"\n✅ Đã lưu toàn bộ file log tại: {save_dir}")

▶ Đang khởi tạo các mô hình và API... (Vui lòng đợi)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: /root/.cache/refined/roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: /root/.cache/refined/roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...


[Serper API] Đã chuyển sang API Key số 4


Tra Serper Web: 100%|██████████| 5403/5403 [17:52<00:00,  5.04it/s]



▶ PHASE 4: LLM dự đoán nhãn Web (Batch=50, Threads=5)


LLM (Web): 100%|██████████| 109/109 [28:11<00:00, 15.52s/it]


▶ PHASE 5: Đánh giá độ chính xác (Accuracy)

🏆 TỔNG KẾT ĐÁNH GIÁ TRÊN FACTKG
Tổng số mẫu đã test: 9041
Tổng độ chính xác (Overall Accuracy): 76.65%

Độ chính xác theo từng loại lập luận (Reasoning Types):
 - Coll:model: 77.48% (3591/4635)
 - Existence: 72.36% (940/1299)
 - Written: 76.91% (2654/3451)
 - Num1: 84.51% (2008/2376)
 - Substitution: 93.84% (3474/3702)
 - Num2: 75.29% (2105/2796)
 - Multi claim: 80.69% (2657/3293)
 - Num3: 73.77% (1150/1559)
 - Num4: 71.91% (727/1011)
 - Multi hop: 63.92% (1325/2073)
 - Question: 79.90% (481/602)
 - Coll:presup: 71.73% (685/955)
 - Negation: 64.46% (847/1314)

✅ Đã lưu toàn bộ file log tại: /content/drive/MyDrive/Data FactKG
